# Batched Voigt-peak fitting with hand-rolled Levenberg-Marquardt (PyTorch / GPU)

This notebook fits a sum of (pseudo-)Voigt peaks to many spectra **simultaneously**,
which is the situation you have with a hyperspectral image: thousands of pixels,
each a spectrum of 600-2000 points, each to be fit with 50-200 peaks.

## Why "residual vector, not scalar loss"

If you fit with autograd + Adam/SGD on a scalar loss `L = sum(r**2)`, you only ever
see `dL/dp`, a single gradient vector. Gauss-Newton / Levenberg-Marquardt instead need
the **Jacobian of the residual itself**, `J = dr/dp` (an `N x P` matrix, `N` = number of
data points, `P` = number of parameters), because the LM step solves

```
(JᵀJ + λI) δ = -Jᵀr
```

This converges in a handful of iterations on smooth least-squares problems like
ours (mixtures of smooth peak shapes), versus hundreds-to-thousands of iterations
for first-order gradient descent. The cost is that you must build a Jacobian
every iteration — but for a single spectrum this is cheap, and for *many* spectra
at once it is exactly the kind of regular, dense linear-algebra workload a GPU wants.

## Why batching + `torch.func` (`vmap` + `jacrev`)

* `vmap` lets us write the residual function for **one** spectrum and automatically
  obtain a function that handles a whole batch, by mapping over a leading batch axis
  without writing a Python loop.
* `jacrev` gives us the exact per-spectrum Jacobian via reverse-mode autodiff (cheap
  here since the residual is a vector of size N << number of scalar backward passes
  it would take element-by-element).
* Composing `vmap(jacrev(f))` gives a function that returns a `(B, N, P)` Jacobian
  for a batch of `B` spectra in one shot, with all the batch arithmetic happening as
  large dense GPU tensor ops, not a Python `for` loop over pixels.

## Why warm-starting helps

LM (like Newton's method) converges fastest when the initial guess is close to the
solution. In a hyperspectral image, neighbouring pixels usually have very similar
spectra (slowly varying chemistry/optics), so the converged parameters of pixel
`i` are an excellent initial guess for pixel `i+1`. We demonstrate that warm-starting
along a raster scan cuts the number of LM iterations (and therefore wall-time)
dramatically compared to a generic "cold" initial guess applied to every pixel.


In [ ]:
import math
import time
import numpy as np
import torch
from torch.func import vmap, jacrev
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Set DTYPE = torch.float64 for higher precision at the cost of ~2x memory/compute.
DTYPE = torch.float32


## 1. The Voigt line shape

The Voigt profile is the convolution of a Gaussian (width `sigma`) and a Lorentzian
(HWHM `gamma`). It has a closed form via the Faddeeva function `w(z)`, which we
evaluate with the Weideman rational/continued-fraction approximation below
(accurate, fully vectorized, and differentiable since it's built only from
elementary complex-tensor ops — required so `torch.func.jacrev` can differentiate
straight through it).


In [ ]:
def weideman_coeffs(N=16):
    '''One-time setup. Returns (coeffs, L). N=16 -> float32-grade; N=24 -> ~1e-10.'''
    M, M2 = 2 * N, 4 * N
    k = np.arange(-M + 1, M)
    L = np.sqrt(N / np.sqrt(2.0))
    t = L * np.tan(k * np.pi / (2 * M))
    f = np.concatenate([[0.0], np.exp(-t**2) * (L**2 + t**2)])
    a = np.real(np.fft.fft(np.fft.fftshift(f))) / M2
    a = np.flipud(a[1:N + 1]).copy()         # highest power first
    return torch.from_numpy(a), float(L)


def faddeeva_w(z, a, L):
    '''Faddeeva w(z) for Im(z) >= 0. z: complex tensor; a,L from weideman_coeffs.'''
    a = a.to(dtype=z.real.dtype)
    iz = 1j * z
    Z = (L + iz) / (L - iz)
    p = torch.zeros_like(Z)
    for c in a:                              # Horner over ~N coeffs (not over data)
        p = p * Z + c
    inv = 1.0 / (L - iz)
    return 2.0 * p * inv * inv + (1.0 / np.sqrt(np.pi)) * inv


def voigt(x, x0, sigma, gamma, a, L):
    '''Area-normalized Voigt. sigma = Gaussian width, gamma = Lorentzian HWHM.'''
    z = ((x - x0) + 1j * gamma) / (sigma * np.sqrt(2.0))
    return faddeeva_w(z, a, L).real / (sigma * np.sqrt(2.0 * np.pi))


# Precompute the Weideman coefficients once. The Horner loop only runs N times,
# independent of data size or batch size, so this is cheap either way -- but every
# Horner step is itself differentiated by jacrev, so a larger N does add some cost
# to every LM iteration. N=16 is float32-grade accuracy and is the default here for
# speed; bump to N=24 (~1e-10 accuracy) if you need higher precision and can afford it.
_WEIDEMAN_N = 16
A_COEFFS, L_CONST = weideman_coeffs(_WEIDEMAN_N)
A_COEFFS = A_COEFFS.to(DEVICE)


## 2. Parametrization

We never let the optimizer see a constrained parameter directly. Instead every
peak has 4 *raw* (unconstrained, `(-inf, inf)`) numbers that get mapped through a
smooth, differentiable transform into the physically-meaningful, constrained
parameter:

| raw param        | transform                                  | physical meaning                  |
|-------------------|---------------------------------------------|------------------------------------|
| `raw_amp`         | `amplitude = softplus(raw_amp)`              | peak area, >= 0                    |
| `raw_center`      | `center = lo + (hi-lo) * sigmoid(raw_center)`| peak position, soft-bounded window |
| `raw_width`       | `width = sigmoid(raw_width) * width_max(center)` | total width, bounded above by a *function of center* |
| `raw_eta`         | `eta = sigmoid(raw_eta)`                     | Gaussian/Lorentzian mix, in [0,1]  |

The Voigt function above wants a `sigma` (Gaussian) and `gamma` (Lorentzian HWHM)
separately, not a single width + mixing fraction. We convert the single `width`/`eta`
pair into `(sigma, gamma)` with

```
sigma = width * (1 - eta) + width_floor
gamma = width * eta
```

so `eta=0` gives a (near) pure Gaussian and `eta=1` gives a (near) pure Lorentzian,
while always staying a valid (non-degenerate) Voigt — a small `width_floor` keeps
`sigma` from hitting exactly zero, which would blow up the `1/sigma` in `voigt()`.
This keeps a single intuitive "total width" knob per peak while still using the
true convolutional Voigt shape (not a linear pseudo-Voigt sum), since that's the
line shape the provided `voigt()` implements.

`width_max(center)` is a user-supplied callable — physically this encodes "peaks
near the edge/center of my spectral window are not allowed to be wider than X".
The default is linear in `center`.


In [ ]:
def default_width_max_fn(x_min, x_max, frac_min=0.01, frac_max=0.08):
    '''Returns f(center) -> max width, linear in center.

    At x_min the max allowed width is frac_min * range; at x_max it is
    frac_max * range. Swap in any callable f(center_tensor) -> width_tensor here.
    '''
    rng = x_max - x_min
    w_lo, w_hi = frac_min * rng, frac_max * rng

    def f(center):
        t = (center - x_min) / rng
        t = torch.clamp(t, 0.0, 1.0)
        return w_lo + (w_hi - w_lo) * t

    return f


WIDTH_FLOOR_FRAC = 1e-3  # sigma = width*(1-eta) + WIDTH_FLOOR_FRAC*width_max, avoids sigma -> 0


def unpack_params(raw, center_lo, center_hi, width_max_fn):
    '''raw: (..., n_peaks, 4) -> dict of physical params, each (..., n_peaks).

    Works for an arbitrary number of leading batch dims (none, or one for vmap),
    since every op below is elementwise / broadcasting.
    '''
    raw_amp, raw_center, raw_width, raw_eta = raw.unbind(dim=-1)

    amplitude = torch.nn.functional.softplus(raw_amp)
    center = center_lo + (center_hi - center_lo) * torch.sigmoid(raw_center)
    w_max = width_max_fn(center)
    frac = torch.sigmoid(raw_width)
    width = frac * w_max
    eta = torch.sigmoid(raw_eta)

    sigma = width * (1.0 - eta) + WIDTH_FLOOR_FRAC * w_max
    gamma = width * eta
    return amplitude, center, sigma, gamma


def voigt_sum_model(raw, x, center_lo, center_hi, width_max_fn, a, L):
    '''Single spectrum: raw (n_peaks, 4) flattened or not, x (N,) -> y_hat (N,).'''
    raw = raw.view(-1, 4)
    amplitude, center, sigma, gamma = unpack_params(raw, center_lo, center_hi, width_max_fn)
    # Broadcast: x (N,1) against per-peak params (1,n_peaks) -> (N, n_peaks), sum over peaks.
    xx = x.view(-1, 1)
    peaks = amplitude.view(1, -1) * voigt(xx, center.view(1, -1), sigma.view(1, -1),
                                           gamma.view(1, -1), a, L)
    return peaks.sum(dim=-1)


## 3. Synthetic data

We generate a known set of peaks (ground truth), simulate noisy spectra from them,
and also build a small 32x32 "image" whose peak parameters drift smoothly across
the grid (slowly varying chemistry), to demonstrate warm-starting later.


In [ ]:
def make_ground_truth_peaks(n_peaks, x_min, x_max, seed=0):
    rng = np.random.default_rng(seed)
    centers = np.sort(rng.uniform(x_min + 0.05 * (x_max - x_min),
                                   x_max - 0.05 * (x_max - x_min), n_peaks))
    amplitudes = rng.uniform(0.5, 5.0, n_peaks)
    sigmas = rng.uniform(0.01, 0.03, n_peaks) * (x_max - x_min)
    gammas = rng.uniform(0.0, 0.02, n_peaks) * (x_max - x_min)
    return dict(centers=centers, amplitudes=amplitudes, sigmas=sigmas, gammas=gammas)


def synth_spectrum(x, peaks, a, L, noise_std=0.0, device=DEVICE, dtype=DTYPE):
    x_t = torch.as_tensor(x, device=device, dtype=dtype)
    centers = torch.as_tensor(peaks["centers"], device=device, dtype=dtype)
    amps = torch.as_tensor(peaks["amplitudes"], device=device, dtype=dtype)
    sigmas = torch.as_tensor(peaks["sigmas"], device=device, dtype=dtype)
    gammas = torch.as_tensor(peaks["gammas"], device=device, dtype=dtype)
    xx = x_t.view(-1, 1)
    y = (amps.view(1, -1) *
         voigt(xx, centers.view(1, -1), sigmas.view(1, -1), gammas.view(1, -1), a, L)
         ).sum(dim=-1)
    if noise_std > 0:
        y = y + noise_std * torch.randn_like(y)
    return x_t, y


# NOTE ON SIZES: the values below are deliberately modest (a few hundred points,
# a dozen-ish peaks, a small image) so this notebook finishes end-to-end in a few
# minutes even on a CPU-only machine. The whole point of the batched-LM design is
# that it scales to the realistic regime -- 600-2000 points, 50-200 peaks, thousands
# of pixels -- *given a GPU*; see the "knobs" section at the end for how to scale up.
N_POINTS = 250
N_PEAKS = 12
X_MIN, X_MAX = 0.0, 1000.0

x_grid = np.linspace(X_MIN, X_MAX, N_POINTS)
true_peaks = make_ground_truth_peaks(N_PEAKS, X_MIN, X_MAX, seed=1)
x_t, y_clean = synth_spectrum(x_grid, true_peaks, A_COEFFS, L_CONST, noise_std=0.0)
_, y_noisy = synth_spectrum(x_grid, true_peaks, A_COEFFS, L_CONST, noise_std=0.05)

plt.figure(figsize=(9, 3))
plt.plot(x_grid, y_clean.cpu(), label="clean ground truth")
plt.plot(x_grid, y_noisy.cpu(), ".", ms=2, alpha=0.5, label="noisy 'data'")
plt.legend(); plt.xlabel("x"); plt.ylabel("y"); plt.title("Synthetic single spectrum")
plt.tight_layout(); plt.show()


In [ ]:
# --- A small "hyperspectral image", each pixel with N_PEAKS peaks whose
#     centers/amplitudes drift slowly and smoothly across the grid, plus noise.
#     (8x8 keeps the whole-image cells fast on CPU; bump this up freely on a GPU --
#     see the "knobs" section.) ---

IMG_H, IMG_W = 8, 8

def make_image_ground_truth(n_peaks, x_min, x_max, h, w, seed=2):
    base = make_ground_truth_peaks(n_peaks, x_min, x_max, seed=seed)
    rng = np.random.default_rng(seed + 1)
    # Smooth per-pixel drift fields via low-frequency random sinusoids.
    yy, xx = np.meshgrid(np.linspace(0, 1, h), np.linspace(0, 1, w), indexing="ij")

    def smooth_field(scale):
        freq = rng.uniform(0.5, 1.5, 2)
        phase = rng.uniform(0, 2 * np.pi, 2)
        return scale * (np.sin(2 * np.pi * freq[0] * xx + phase[0]) +
                         np.sin(2 * np.pi * freq[1] * yy + phase[1]))

    center_drift = smooth_field(0.01 * (x_max - x_min))[..., None] * np.linspace(-1, 1, n_peaks)
    amp_drift = 1.0 + 0.2 * smooth_field(1.0)[..., None]

    centers_img = base["centers"][None, None, :] + center_drift          # (h, w, n_peaks)
    amps_img = base["amplitudes"][None, None, :] * amp_drift             # (h, w, n_peaks)
    sigmas_img = np.broadcast_to(base["sigmas"], (h, w, n_peaks)).copy()
    gammas_img = np.broadcast_to(base["gammas"], (h, w, n_peaks)).copy()
    return centers_img, amps_img, sigmas_img, gammas_img


centers_img, amps_img, sigmas_img, gammas_img = make_image_ground_truth(
    N_PEAKS, X_MIN, X_MAX, IMG_H, IMG_W)

x_t = torch.as_tensor(x_grid, device=DEVICE, dtype=DTYPE)


def render_image(centers_img, amps_img, sigmas_img, gammas_img, x_t, a, L, noise_std,
                  device=DEVICE, dtype=DTYPE):
    h, w, n_peaks = centers_img.shape
    c = torch.as_tensor(centers_img.reshape(-1, n_peaks), device=device, dtype=dtype)
    am = torch.as_tensor(amps_img.reshape(-1, n_peaks), device=device, dtype=dtype)
    si = torch.as_tensor(sigmas_img.reshape(-1, n_peaks), device=device, dtype=dtype)
    ga = torch.as_tensor(gammas_img.reshape(-1, n_peaks), device=device, dtype=dtype)
    # (B, N, n_peaks): broadcast x over batch & peaks.
    xx = x_t.view(1, -1, 1)
    y = (am.view(-1, 1, n_peaks) *
         voigt(xx, c.view(-1, 1, n_peaks), si.view(-1, 1, n_peaks), ga.view(-1, 1, n_peaks), a, L)
         ).sum(dim=-1)
    if noise_std > 0:
        y = y + noise_std * torch.randn_like(y)
    return y.view(h, w, -1)


image_spectra = render_image(centers_img, amps_img, sigmas_img, gammas_img, x_t,
                              A_COEFFS, L_CONST, noise_std=0.05)
print("image_spectra shape (H, W, N_points):", image_spectra.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (i, j) in zip(axes, [(0, 0), (IMG_H // 2, IMG_W // 2), (IMG_H - 1, IMG_W - 1)]):
    ax.plot(x_grid, image_spectra[i, j].cpu())
    ax.set_title(f"pixel ({i},{j})")
plt.tight_layout(); plt.show()


## 4. The batched Levenberg-Marquardt solver

The residual function below is written for a **single** spectrum (no batch
dimension at all). We then build:

* `batched_residual = vmap(residual_fn)` — maps the same function over a batch of
  `(raw_params, y)` pairs (the shared `x`, bounds, and Faddeeva coefficients are
  passed with `in_dims=None`, i.e. "broadcast, don't map").
* `batched_jacobian = vmap(jacrev(residual_fn))` — `jacrev` differentiates the
  *vector-valued* residual of one spectrum w.r.t. its raw parameters, giving an
  `(N, P)` matrix; `vmap` then stacks that over the batch into `(B, N, P)`, again
  with no explicit batch-handling code inside `residual_fn` itself.

Each LM iteration is then pure batched dense linear algebra:

```
J        : (B, N, P)
r        : (B, N)
JtJ      : (B, P, P)  = J^T J
Jtr      : (B, P)     = J^T r
A        : (B, P, P)  = JtJ + lambda * diag(diag(JtJ))      (Marquardt scaling)
delta    : (B, P)     solves  A @ delta = -Jtr
```

solved with one `torch.linalg.solve` call for the whole batch — no Python loop
over pixels anywhere in the hot path.


In [ ]:
def make_residual_fn(x, center_lo, center_hi, width_max_fn, a, L):
    '''Closure capturing everything that is shared across the batch
    (x grid, per-peak center bounds, the width_max callable, Faddeeva coeffs).
    Returns residual_fn(raw_flat, y) -> (N,) for a SINGLE spectrum.
    '''
    def residual_fn(raw_flat, y):
        y_hat = voigt_sum_model(raw_flat, x, center_lo, center_hi, width_max_fn, a, L)
        return y_hat - y
    return residual_fn


class BatchedVoigtLM:
    '''Hand-rolled batched Levenberg-Marquardt for sums of Voigt peaks.

    All `n_fits` independent least-squares problems are solved in lock-step:
    every iteration does one Jacobian build + one batched linear solve. Fits
    that have already converged are frozen (their delta is zeroed) so they
    stop changing while the rest keep iterating.
    '''

    def __init__(self, x, n_peaks, center_windows, width_max_fn,
                 a=A_COEFFS, L=L_CONST, device=DEVICE, dtype=DTYPE,
                 lambda_init=1e-2, lambda_up=3.0, lambda_down=2.0,
                 lambda_min=1e-9, lambda_max=1e9,
                 cost_tol=1e-10, step_tol=1e-8, max_iters=50):
        self.x = torch.as_tensor(x, device=device, dtype=dtype)
        self.n_peaks = n_peaks
        self.n_params = n_peaks * 4
        # center_windows: (n_peaks, 2) array of (lo, hi) soft bounds, shared across batch.
        cw = torch.as_tensor(center_windows, device=device, dtype=dtype)
        self.center_lo, self.center_hi = cw[:, 0], cw[:, 1]
        self.width_max_fn = width_max_fn
        self.a, self.L = a.to(device=device), L
        self.device, self.dtype = device, dtype
        self.lambda_init, self.lambda_up, self.lambda_down = lambda_init, lambda_up, lambda_down
        self.lambda_min, self.lambda_max = lambda_min, lambda_max
        self.cost_tol, self.step_tol, self.max_iters = cost_tol, step_tol, max_iters

        residual_fn = make_residual_fn(self.x, self.center_lo, self.center_hi,
                                        self.width_max_fn, self.a, self.L)
        self._residual_fn = residual_fn
        # in_dims=(0, 0): map both raw params and y over the batch dimension (dim 0).
        self._batched_residual = vmap(residual_fn, in_dims=(0, 0))
        # jacrev differentiates a single spectrum's (N,) residual w.r.t. its (P,) params
        # -> (N, P) Jacobian; vmap then stacks that over the batch -> (B, N, P).
        self._batched_jacobian = vmap(jacrev(residual_fn, argnums=0), in_dims=(0, 0))

    def fit(self, y_batch, raw_init, lam_init=None, return_history=False):
        '''y_batch: (B, N). raw_init: (B, n_peaks, 4) or (B, n_params).
        Returns dict with converged params, iteration counts, final cost, etc.
        '''
        B = y_batch.shape[0]
        raw = raw_init.reshape(B, self.n_params).clone().to(self.device, self.dtype)
        y_batch = y_batch.to(self.device, self.dtype)

        lam = torch.full((B,), self.lambda_init, device=self.device, dtype=self.dtype)
        if lam_init is not None:
            lam = lam_init.clone()
        converged = torch.zeros(B, dtype=torch.bool, device=self.device)
        iters_used = torch.zeros(B, dtype=torch.long, device=self.device)

        r = self._batched_residual(raw, y_batch)              # (B, N)
        cost = (r ** 2).sum(dim=-1)                            # (B,)
        history = [] if return_history else None

        eye = torch.eye(self.n_params, device=self.device, dtype=self.dtype)

        n_iter_done = 0
        for it in range(self.max_iters):
            n_iter_done = it + 1
            active = ~converged
            if not active.any():
                break

            J = self._batched_jacobian(raw, y_batch)           # (B, N, P)
            r = self._batched_residual(raw, y_batch)           # (B, N)
            JtJ = torch.einsum("bni,bnj->bij", J, J)            # (B, P, P)
            Jtr = torch.einsum("bni,bn->bi", J, r)              # (B, P)

            diag_JtJ = torch.diagonal(JtJ, dim1=-2, dim2=-1).clamp_min(1e-12)  # (B, P)
            damping = lam.view(-1, 1) * diag_JtJ                 # Marquardt scaling
            A = JtJ + damping.unsqueeze(-1) * eye                # (B, P, P)

            # Batched solve for the LM step. (Swap for cholesky_solve(cholesky(A)) if you
            # need extra speed and are confident A stays well-conditioned/SPD.)
            delta = torch.linalg.solve(A, -Jtr.unsqueeze(-1)).squeeze(-1)   # (B, P)
            delta = torch.where(active.unsqueeze(-1), delta, torch.zeros_like(delta))

            raw_trial = raw + delta
            r_trial = self._batched_residual(raw_trial, y_batch)
            cost_trial = (r_trial ** 2).sum(dim=-1)

            success = (cost_trial < cost) & active

            raw = torch.where(success.unsqueeze(-1), raw_trial, raw)
            cost = torch.where(success, cost_trial, cost)
            lam = torch.where(success, (lam / self.lambda_down).clamp_min(self.lambda_min),
                               (lam * self.lambda_up).clamp_max(self.lambda_max))

            step_norm = delta.norm(dim=-1)
            rel_cost_drop = (cost_trial - cost).abs() / cost.clamp_min(1e-30)
            newly_converged = active & success & (
                (step_norm < self.step_tol) | (rel_cost_drop < self.cost_tol))
            iters_used = torch.where(active, torch.full_like(iters_used, it + 1), iters_used)
            converged = converged | newly_converged

            if return_history:
                history.append(cost.detach().clone())

        result = dict(raw=raw, cost=cost, converged=converged, iters_used=iters_used,
                       lam=lam, n_iter_done=n_iter_done)
        if return_history:
            result["history"] = torch.stack(history, dim=1)  # (B, n_iter_done)
        return result

    def physical_params(self, raw):
        '''raw: (..., n_params) -> dict of (..., n_peaks) physical params.'''
        raw = raw.reshape(*raw.shape[:-1], self.n_peaks, 4)
        amplitude, center, sigma, gamma = unpack_params(raw, self.center_lo, self.center_hi,
                                                          self.width_max_fn)
        return dict(amplitude=amplitude, center=center, sigma=sigma, gamma=gamma)

    def predict(self, raw):
        '''raw: (..., n_params) -> (..., N) model spectra.'''
        flat = raw.reshape(-1, self.n_params)
        preds = torch.stack([
            voigt_sum_model(flat[i], self.x, self.center_lo, self.center_hi,
                             self.width_max_fn, self.a, self.L)
            for i in range(flat.shape[0])
        ])
        return preds.reshape(*raw.shape[:-1], -1)


**Note on the predict() loop above:** that one *is* a Python loop, but it is only
ever used to render fitted curves for plotting/inspection after the fact — never inside
the optimization hot path. (You could equally `vmap` it; it's left as a simple loop
since it runs once, on converged results, not B*max_iters times.)

**Note on Jacobian sparsity:** each peak only meaningfully affects points near its
center, so `J` is column-block-sparse in a structured sense. At the problem sizes
here (N up to ~2000, P up to ~800) the dense `(N,P)` Jacobian and `(P,P)` normal
equations comfortably fit in GPU memory and the dense batched solve is bandwidth-bound
in a GPU-friendly way, so we use dense solves by default. If you push to much larger P
(thousands of peaks) it would be worth exploiting the banded/block structure of `J`
(e.g. only computing Jacobian columns for peaks within a window of each point) to
cut the `O(N P^2)` cost of forming `JtJ`.


## 5. Single-spectrum fit: validate against ground truth

We fit the single noisy spectrum generated above, starting from a generic
"spread peaks evenly, small amplitude" initial guess (no warm start), and compare
recovered vs. true peak parameters.


In [ ]:
def generic_init(n_peaks, x_min, x_max, batch_size, device=DEVICE, dtype=DTYPE):
    '''A simple, generic cold-start initial guess: peaks spread evenly across
    the window, modest amplitude, modest width, eta=0.5. Returns raw (B, n_peaks, 4).
    '''
    centers0 = torch.linspace(x_min, x_max, n_peaks + 2, device=device, dtype=dtype)[1:-1]
    raw_amp0 = torch.full((n_peaks,), 0.5, device=device, dtype=dtype)   # softplus(0.5)~0.97
    raw_center0 = torch.zeros(n_peaks, device=device, dtype=dtype)        # sigmoid(0)=0.5 -> mid-window
    raw_width0 = torch.zeros(n_peaks, device=device, dtype=dtype)         # sigmoid(0)=0.5 -> half of width_max
    raw_eta0 = torch.zeros(n_peaks, device=device, dtype=dtype)           # sigmoid(0)=0.5

    raw = torch.stack([raw_amp0, raw_center0, raw_width0, raw_eta0], dim=-1)  # (n_peaks, 4)
    return centers0, raw.unsqueeze(0).repeat(batch_size, 1, 1)


width_max_fn = default_width_max_fn(X_MIN, X_MAX)

# Center windows: give each (evenly spaced) initial center a window wide enough to
# reach any true peak, since raw_center=0 maps to the window midpoint by construction.
init_centers, raw0 = generic_init(N_PEAKS, X_MIN, X_MAX, batch_size=1)
half_window = (X_MAX - X_MIN) / N_PEAKS * 1.5
center_windows = torch.stack([init_centers - half_window, init_centers + half_window], dim=-1)

solver = BatchedVoigtLM(x_grid, N_PEAKS, center_windows.cpu().numpy(), width_max_fn)

t0 = time.time()
res = solver.fit(y_noisy.unsqueeze(0), raw0, return_history=True)
torch.cuda.synchronize() if DEVICE.type == "cuda" else None
t1 = time.time()
print(f"Single spectrum fit: {res['iters_used'].item()} iterations, {t1 - t0:.4f} s, "
      f"final cost {res['cost'].item():.3e}")

y_fit = solver.predict(res["raw"])[0]

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                          gridspec_kw=dict(height_ratios=[3, 1]))
axes[0].plot(x_grid, y_noisy.cpu(), ".", ms=2, alpha=0.4, label="data")
axes[0].plot(x_grid, y_fit.detach().cpu(), label="fit")
axes[0].legend(); axes[0].set_title("Single-spectrum fit")
axes[1].plot(x_grid, (y_fit.detach() - y_noisy).cpu())
axes[1].set_ylabel("residual"); axes[1].set_xlabel("x")
plt.tight_layout(); plt.show()


In [ ]:
# Compare recovered vs. true peaks by nearest-center matching (order isn't preserved).
fitted = solver.physical_params(res["raw"])
fit_centers = fitted["center"][0].detach().cpu().numpy()
fit_amps = fitted["amplitude"][0].detach().cpu().numpy()
fit_sigmas = fitted["sigma"][0].detach().cpu().numpy()
fit_gammas = fitted["gamma"][0].detach().cpu().numpy()

true_centers = true_peaks["centers"]
order = np.argsort(true_centers)
rows = []
for k in order[:15]:  # show first 15 for brevity
    j = np.argmin(np.abs(fit_centers - true_centers[k]))
    rows.append((true_centers[k], fit_centers[j],
                 true_peaks["amplitudes"][k], fit_amps[j],
                 true_peaks["sigmas"][k], fit_sigmas[j],
                 true_peaks["gammas"][k], fit_gammas[j]))

print(f"{'true c':>8} {'fit c':>8} {'true A':>8} {'fit A':>8} "
      f"{'true sig':>9} {'fit sig':>9} {'true gam':>9} {'fit gam':>9}")
for r in rows:
    print(" ".join(f"{v:8.3f}" for v in r))


## 6. Whole-image fit: scaling, and cold-start vs. warm-start

We fit all `IMG_H * IMG_W` pixel spectra at once (a single batched LM solve) for
the cold-start timing, then refit a **raster-scanned chain with warm-starting**:
each pixel's initial guess is the previous pixel's converged `raw` parameters.

To isolate "how many iterations does warm vs. cold starting need" from "how many
pixels are we fitting", the chain below is solved one pixel at a time (batch size 1
per step) over a small subset (`N_CHAIN_DEMO` pixels) of the raster order, **not**
the whole image -- a Python loop of single-pixel LM solves is the slowest possible
way to use this code (every step pays full per-call overhead for a batch of 1) and
is only meant to demonstrate the *iteration-count* effect cleanly. The very next
cell shows the throughput-correct pattern instead: warm-starting a whole *row*
(a batch of `IMG_W` pixels) at once from the row above.


In [ ]:
B = IMG_H * IMG_W
y_image_flat = image_spectra.reshape(B, -1)   # (B, N)

_, raw0_image = generic_init(N_PEAKS, X_MIN, X_MAX, batch_size=B)

t0 = time.time()
res_cold = solver.fit(y_image_flat, raw0_image)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
t1 = time.time()
cold_time = t1 - t0
cold_iters = res_cold["iters_used"].float()
print(f"COLD start, full image as one batch of {B}: {cold_time:.3f} s, "
      f"mean iters {cold_iters.mean():.1f}, max iters {cold_iters.max():.0f}, "
      f"converged {res_cold['converged'].float().mean()*100:.1f}%")


In [ ]:
# --- Warm-started raster scan over a SMALL SUBSET of pixels: each pixel's init
# is the previous pixel's converged raw. We deliberately only chain N_CHAIN_DEMO
# pixels (not the full B): solving one-pixel-at-a-time in a Python loop is the
# slowest possible use of this code (no batching at all, full per-call overhead
# every step), so it's kept small and is only here to show the iteration-count
# effect, not to be a throughput benchmark.

N_CHAIN_DEMO = min(12, B)
raster_y = y_image_flat[:N_CHAIN_DEMO]  # already raster-ordered (row-major), (N_CHAIN_DEMO, N)

raw_prev = raw0_image[0:1]  # generic cold init for the very first pixel only
warm_iters = torch.zeros(N_CHAIN_DEMO, dtype=torch.long, device=DEVICE)
warm_raw = torch.zeros(N_CHAIN_DEMO, solver.n_params, device=DEVICE, dtype=DTYPE)

t0 = time.time()
for i in range(N_CHAIN_DEMO):
    y_i = raster_y[i:i + 1]
    r = solver.fit(y_i, raw_prev)
    warm_iters[i] = r["iters_used"][0]
    warm_raw[i] = r["raw"][0]
    raw_prev = r["raw"]  # warm-start the next pixel from this one's converged solution
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
t1 = time.time()
warm_time_chain = t1 - t0

cold_iters_subset = cold_iters[:N_CHAIN_DEMO]
print(f"WARM start, sequential chain over {N_CHAIN_DEMO} pixels (batch size 1 each): "
      f"{warm_time_chain:.3f} s, mean iters {warm_iters.float().mean():.1f}, "
      f"max iters {warm_iters.max().item()}")
print(f"COLD start mean iters (same {N_CHAIN_DEMO} pixels, from the full-image batch "
      f"fit above): {cold_iters_subset.mean():.1f}  ->  "
      f"WARM start mean iters: {warm_iters.float().mean():.1f}  "
      f"({(1 - warm_iters.float().mean()/cold_iters_subset.mean())*100:.0f}% fewer)")


The sequential chain above intentionally uses **batch size 1 per step** so the
*iteration-count* comparison is apples-to-apples (it isolates "warm vs. cold start
quality", not "many-small-fits vs. one-big-fit" GPU efficiency, which is a separate
effect demonstrated by the next cell). In production you'd warm-start a whole
*batch* at once — e.g. process the image row by row, initializing row `r` from the
converged result of row `r-1` for every pixel in that row in parallel — to get both
benefits (fewer iterations *and* full GPU batch utilization) simultaneously:


In [ ]:
# --- Practical pattern: warm-start row-by-row, batched across each row. ---

def fit_image_warm_rows(solver, y_image, raw_init_row0, return_iters=False):
    '''y_image: (H, W, N). raw_init_row0: (W, n_peaks, 4) init for row 0.
    Every later row is warm-started from the row above (same column).
    '''
    H, W, N = y_image.shape
    raw_prev_row = raw_init_row0.reshape(W, -1)
    all_raw = torch.zeros(H, W, solver.n_params, device=solver.device, dtype=solver.dtype)
    all_iters = torch.zeros(H, W, dtype=torch.long, device=solver.device)
    for row in range(H):
        y_row = y_image[row].reshape(W, N)
        r = solver.fit(y_row, raw_prev_row)
        all_raw[row] = r["raw"]
        all_iters[row] = r["iters_used"]
        raw_prev_row = r["raw"]
    out = (all_raw, all_iters) if return_iters else all_raw
    return out


def fit_image_cold_rows(solver, y_image, raw_init_row0, return_iters=False):
    '''Same row-batched call pattern as fit_image_warm_rows, but every row gets
    a fresh generic (cold) init instead of the row above's converged params --
    this isolates the warm-vs-cold *iteration* effect at matched call granularity,
    since splitting one big batch into H smaller per-row calls has its own
    (CPU tracing) overhead that would otherwise confound the comparison.
    '''
    H, W, N = y_image.shape
    all_raw = torch.zeros(H, W, solver.n_params, device=solver.device, dtype=solver.dtype)
    all_iters = torch.zeros(H, W, dtype=torch.long, device=solver.device)
    for row in range(H):
        y_row = y_image[row].reshape(W, N)
        r = solver.fit(y_row, raw_init_row0.reshape(W, -1))
        all_raw[row] = r["raw"]
        all_iters[row] = r["iters_used"]
    out = (all_raw, all_iters) if return_iters else all_raw
    return out


_, raw_init_row0 = generic_init(N_PEAKS, X_MIN, X_MAX, batch_size=IMG_W)

t0 = time.time()
raw_cold_rows, iters_cold_rows = fit_image_cold_rows(solver, image_spectra, raw_init_row0,
                                                       return_iters=True)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
cold_rows_time = time.time() - t0

t0 = time.time()
raw_warm_rows, iters_warm_rows = fit_image_warm_rows(solver, image_spectra, raw_init_row0,
                                                       return_iters=True)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
warm_rows_time = time.time() - t0

print("Matched call granularity (both as H batched fits of size W):")
print(f"  cold rows (fresh generic init every row): {cold_rows_time:8.3f} s, "
      f"{iters_cold_rows.float().mean():5.1f} iters/pixel avg")
print(f"  warm rows (init from row above):          {warm_rows_time:8.3f} s, "
      f"{iters_warm_rows.float().mean():5.1f} iters/pixel avg")
print()
print("For reference, one single big batched call (no row-splitting overhead):")
print(f"  cold, single batch of {B}:                 {cold_time:8.3f} s, "
      f"{cold_iters.mean():5.1f} iters/pixel avg")


**A note on what you'll actually see above.** The single-pixel chain in the
previous section gives a clean result because consecutive pixels there are *nearly
identical* fitting problems (same peaks, tiny center drift), so the previous
solution is an excellent initial guess. Row-to-row warm-starting is a harder test:
each row is a materially different multi-peak problem, and with `N_PEAKS` peaks
that overlap, the residual surface has many near-degenerate local minima (e.g. a
peak collapsing onto a neighboring one, or shrinking to a narrow spike that locally
soaks up residual). The previous row's converged solution can land *inside the
basin of one of those local minima* instead of the basin of the true global
solution that fits the new row best -- and escaping a bad basin can take more LM
iterations than just restarting from a neutral, generic guess. This is a genuine
property of warm-starting nonconvex least-squares problems, not a bug in the
solver: warm-starting helps when consecutive problems are similar enough that the
old solution sits in the right basin (as in the chain demo), and can hurt when
they're different enough that it doesn't (as can happen row-to-row here). In
practice, look at `iters_warm_rows` per-pixel below to see *where* this happens --
it's typically isolated to a handful of pixels, not the whole image.


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(iters_warm_rows.cpu(), cmap="viridis")
plt.colorbar(label="LM iterations to converge")
plt.title("Per-pixel iteration count, row-warm-started fit")
plt.xlabel("column"); plt.ylabel("row")
plt.tight_layout(); plt.show()

# Sanity-check fit quality on a couple of pixels from the warm row-batched fit.
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (i, j) in zip(axes, [(0, 0), (IMG_H - 1, IMG_W - 1)]):
    y_true = image_spectra[i, j].cpu()
    y_hat = solver.predict(raw_warm_rows[i, j]).detach().cpu()
    ax.plot(x_grid, y_true, ".", ms=2, alpha=0.4, label="data")
    ax.plot(x_grid, y_hat, label="fit")
    ax.set_title(f"pixel ({i},{j})"); ax.legend()
plt.tight_layout(); plt.show()


## 7. Knobs

* **Number of peaks (`N_PEAKS`) / points (`N_POINTS`)**: just change the constants
  in section 3 and rerun; `BatchedVoigtLM` doesn't hard-code either. Memory scales
  roughly as `O(B * N * P)` for the Jacobian and `O(B * P^2)` for the normal
  equations, where `P = 4 * N_PEAKS`.
* **Tolerances**: `cost_tol` (relative cost-drop convergence) and `step_tol`
  (step-norm convergence) are constructor args to `BatchedVoigtLM`. `max_iters`
  caps the worst case; fits that haven't converged by then are returned with
  `converged=False` so you can inspect/re-run them with different inits.
* **Lambda (damping) schedule**: `lambda_init`, `lambda_up`, `lambda_down`,
  `lambda_min`, `lambda_max` — the defaults (`up=3, down=2`) are a standard
  conservative LM schedule; more aggressive `down` (e.g. 3-5) speeds convergence
  on easy/smooth problems but can overshoot on hard ones.
* **`width_max_fn`**: swap in any `callable(center_tensor) -> width_tensor`. The
  default (`default_width_max_fn`) is linear in `center`; e.g. for spectra where
  peaks are uniformly narrow, just use a constant function `lambda c: torch.full_like(c, w)`.
* **`center_windows`**: per-peak soft-bound windows passed to `BatchedVoigtLM`;
  tighten them if peaks are jumping onto the wrong feature, widen them if a peak's
  true location is outside its window (it will saturate the sigmoid and stop moving).
* **`DTYPE`**: flip the global `DTYPE = torch.float32` (top of the notebook, section 0)
  to `torch.float64` for higher precision at ~2x memory/compute cost; everything
  downstream (model, solver) is dtype-agnostic and just inherits it.
* **Jacobian sparsity**: not exploited by default (see note in section 4) — for
  very large `N_PEAKS` (thousands), restricting each peak's Jacobian columns to a
  local window around its center would cut the `O(N P^2)` cost of `JtJ`.
